# Dating admixture, and selection in admixed individuals

**Purpose.** Admixture proportions say *how much*, local ancestry says *where* — this
exercise asks **when**. Recombination breaks ancestry tracts down at a known rate, so the
decay of ancestry linkage disequilibrium with genetic distance dates the event. Four
tools attack this differently, and the last part asks whether selection has acted on the
admixed ancestry since.

**What you will do**
 - run **ALDER** and **MALDER** on the decay of admixture LD, and read off a date
 - run **fastGLOBETROTTER** on ChromoPainter output to date and describe the event
 - run **MOSAIC**, which needs no reference labelling
 - use **AdaptMix** to test whether ancestry proportions at a locus depart from the
   genome-wide average, which is what selection after admixture would look like

**The data.** 16 present-day populations, **256 individuals**, chromosomes 20-22:

| Region | Populations |
|---|---|
| Africa | BantuKenya 11, BantuSouthAfrica 8, Mandenka 22, MbutiPygmy 13 |
| Central South Asia | Balochi 21, Burusho 25, Kalash 23, Makrani 22, Pathan 22 |
| East Asia | HanNchina 10, Mongola 10 |
| Europe | English 6, NorthItalian 12, Orcadian 15, Sardinian 28, Tuscan 8 |

plus a **simulated admixed group of 20 individuals**: 80% Brahui (Pakistan) and 20%
Yoruba (Nigeria), admixing **30 generations ago**. Because it is simulated, the true date
and the true proportions are known — so each tool can be judged against the answer.

**Note.** This exercise **compiles the software from source** as it goes, so several cells
take a few minutes. It continues from
[ChromoPainter and fineSTRUCTURE](chromopainter_finestructure_human.ipynb), which produced
the ChromoPainter output used in part 2.

## Setup

All the paths used by this exercise are set in the cell below.

In [ ]:
#############################################################
# ALL PATHS ARE SET HERE
# If the data moves, this is the ONLY cell you need to change.
# No cell below this one uses a full path.
#############################################################

# where the shared data lives
DATA=/course/data/current_data/dating_admixture

# where you will do the exercise
WORK_DIR=$HOME/dating_admixture

mkdir -p $WORK_DIR
echo $WORK_DIR > $HOME/.dating_admixture_workdir
cd $WORK_DIR

echo --- unpacking the practical files ---
tar -xzf $DATA/AdmixtureSelectionPractical.tar.gz -C .

echo; echo --- folders ---
ls

# 1 Inferring admixture: ALDER/MALDER

Navigate to the folder `AlderMalderFiles/`. First, we will run ALDER to detect admixture in the simulated population.

Unzip and extract ALDER:

In [ ]:
# the practical files were unpacked in the setup cell
# stay inside AlderMalderFiles: every cell below this one works from there,
# and the Globetrotter section later moves across with 'cd ../GlobetrotterFiles'
cd AlderMalderFiles
tar -xzf alder_v1.03.tar.gz
ls


Then compile:

In [ ]:
cd alder; make; cd ..

Then run on the Brahui/Yoruba simulation:

In [ ]:
alder/./alder -p BrahuiYorubaSimulation.alder.par > BrahuiYorubaSimulation.alder.out

**Questions**
 - ALDER fits an exponential decay to the admixture LD. What is on each axis of that decay curve?
 - What date does it infer, and how close is it to the true 30 generations?

The results of the above run will be in `BrahuiYorubaSimulation.alder.out`.

Also, run MALDER on the Brahui/Yoruba simulation. To do so, first unzip and extract
MALDER:

In [ ]:
unzip malder-master.zip

Then compile

In [ ]:
cd malder-master/MALDER; make; cd ../..

Then run on the Brahui/Yoruba simulation:

In [ ]:
malder-master/MALDER/./malder -p BrahuiYorubaSimulation.malder.par > BrahuiYorubaSimulation.malder.out

**Questions**
 - MALDER allows more than one admixture event. Does it find one event or several here?
 - Does the inferred date change when you use different surrogate populations for the sources?

Once finished, answer the following questions:
1. Does ALDER detect admixture in this simulation? If so, what is the inferred date?
2. What does the evidence for admixture look like here?
3. When running MALDER, does the inferred admixture change when using different combinations of the surrogate populations?

# 2 Inferring admixture: fastGLOBETROTTER

Navigate to the folder GlobetrotterFiles/. As mentioned in the lecture, running GLOBETROTTER or fastGLOBETROTTER requires three steps:
1. use CHROMOPAINTER to paint surrogate populations against each other
2. use CHROMOPAINTER to paint target (admixed) populations against surrogates 3. run GLOBETROTTER or fastGLOBTROTTER using combined results from (1)-(2)

For steps (1)-(2), we will use ChromoPainterv2. Unzip and compile ChromoPainterv2:

In [ ]:
cd ../GlobetrotterFiles
tar -xzvf ChromoPainterv2.tar.gz
gcc -Wall -o ChromoPainterv2 ChromoPainterv2.c -lm -lz

We have already done step (1) in the last practical. For step (2), we have also done this in the last practical, but note below I have highlighted how we use `-s 10` here to output painting samples:

In [ ]:
./ChromoPainterv2 -g data/BrahuiYorubaSimulationChrom22.haplotypes \
-r data/BrahuiYorubaSimulationChrom22.recomrates \
-t example/BrahuiYorubaSimulation.idfile.txt \
-f BrahuiYorubaSimulation.poplistReduced.txt 0 0 \
-o example/BrahuiYorubaSimulationAdmixtureChrom22 -s 10

**Question**
 - ChromoPainter paints each admixed individual as a mosaic of the reference haplotypes. Why is that a better input for dating than the raw genotypes?

Repeat the above ChromoPainterv2 command for chromosomes 20 and 21. As mentioned in the lecture, there are two output files of interest for this analysis: `example/BrahuiYorubaSimulationAdmixtureChrom22.chunklengths.out`
and `example/BrahuiYorubaSimulationAdmixtureChrom22.samples.out`.

(In real applications, we want to sum the `.chunklengths.out` files across chromosomes, and then combine the output from steps (1) and (2). For simplicity here, we will use the combined matrix we made in the previous practical, which is only for chromosome 22, in `data/BrahuiYorubaSimulationAllVersusAllChrom22.chunklengths.out`.)

Finally, for step (3) we’ll run fastGLOBETROTTER to infer admixture, using this output from ChromoPainterv2. Unzip and extract fastGLOBETROTTER:

In [ ]:
./ChromoPainterv2 -g data/BrahuiYorubaSimulationChrom20.haplotypes \
-r data/BrahuiYorubaSimulationChrom20.recomrates \
-t example/BrahuiYorubaSimulation.idfile.txt \
-f BrahuiYorubaSimulation.poplistReduced.txt 0 0 \
-o example/BrahuiYorubaSimulationAdmixtureChrom20 -s 10 

In [ ]:
./ChromoPainterv2 -g data/BrahuiYorubaSimulationChrom21.haplotypes \
-r data/BrahuiYorubaSimulationChrom21.recomrates \
-t example/BrahuiYorubaSimulation.idfile.txt \
-f BrahuiYorubaSimulation.poplistReduced.txt 0 0 \
-o example/BrahuiYorubaSimulationAdmixtureChrom21 -s 10

**Question**
 - The same ChromoPainter command is repeated for chromosomes 20, 21 and 22. Why run each chromosome separately rather than all at once?

In [ ]:
tar -xzvf fastGLOBETROTTER.tar.gz

Next compile with:

In [ ]:
R CMD SHLIB -o fastGLOBETROTTERCompanion.so fastGLOBETROTTERCompanion.c -lz

To run fastGLOBETROTTER for the Brahui-Yoruba simulation, type:

In [ ]:
R < fastGLOBETROTTER.R BrahuiYorubaSimulationAdmixture.paramfile.txt BrahuiYorubaSimulationAdmixture.samplesfile.txt BrahuiYorubaSimulationAdmixture.recomfile.txt 1 --no-save > output.out

**Questions**
 - fastGLOBETROTTER reports a date, the proportions, and which populations best describe each source. How do its sources compare with the true Brahui and Yoruba?
 - Does it prefer a single admixture event or multiple?

It will take a few minutes to complete. You can follow progress by typing:

In [ ]:
# follow the progress of the run
tail -n 30 output.out


Once finished, the following output files will be produced, each in the `example/` directory:

`example/BrahuiYorubaSimulationAdmixed.fastGT.main.txt` 
`example/BrahuiYorubaSimulationAdmixed.fastGT.main.pdf` 
`example/BrahuiYorubaSimulationAdmixed.fastGT.main_curves.txt` `example/BrahuiYorubaSimulationAdmixed.fastGT.boot.txt`

Using these files, answer the following questions:
1. From the fastGLOBETROTTER user manual, what do the different measures in BrahuiYorubaSimulationAdmixed.fastGT.main.txt tell you? In particular what is fastGLOBTROTTER’s conclusion about admixture in this application? And what are the inferred sources and dates of the admixture event?
2. How do you interpret the coancestry curves in BrahuiYorubaSimulationAdmixed.fastGT.main.pdf? Do the results from BrahuiYorubaSimulationAdmixed.fastGT.main.txt make sense in light of these coancestry curves?
3. How confident are the date estimates?

# 3 Inferring admixture: MOSAIC

Navigate to the folder `MosaicFiles/`. Then unzip and extract MOSAIC:

In [ ]:
cd ../MosaicFiles/
tar -xzvf mosaic-master.tar.gz

Then run on the Brahui/Yoruba simulation:

In [ ]:
Rscript mosaic-master/mosaic.R -c 20:22 -p "Balochi BantuKenya BantuSouthAfrica
Burusho English HanNchina Kalash Makrani Mandenka MbutiPygmy Mongola NorthItalian
Orcadian Pathan Sardinian Tuscan" BrahuiYorubaSimulation -a 2 data/

**Questions**
 - MOSAIC is given no labelling of which reference individuals belong to which population. How does that change what it can and cannot tell you?
 - `-a 2` asks for two ancestries. What would happen with `-a 3`?

It will take a few minutes to complete. The results of the above run will be in three folders: `MOSAIC_RESULTS`, `MOSAIC_PLOTS`, `FREQS`).
Looking at the plots in `MOSAIC_PLOTS/`, answer the following questions:
1. What are the conclusions of admixture here, i.e. the inferred date and sources?
2. Does it seem as if the algorithm has converged? 
3. What does the local painting look like?

# 4 Inferring selection in admixed inds: ADAPTMIX

For this last section, we will simulate and test for selection in admixed populations with AdaptMix, using example data provided with the program. This data is comprised of a small subset of data from 1000 Genomes populations. In particular we will test for selection in a simulated admixed Peruvian population (PEL), using admixture surrogates from China (CHB), Nigeria (YRI) and Spain (IBS).

Navigate to the folder `AdaptMixFiles/`. Then unzip and extract AdaptMix and AdaptMixSimulator:

In [ ]:
cd ../AdaptMixFiles
tar -xzvf AdaptMixv1.tar.gz
tar -xzvf AdaptMixSimulator.tar.gz

First we will use `AdaptMixSimulator.R`, running it with `CHB_selection_paramfile.txt` and the example data in `simexample/`, to generate a simulated “`PEL`” population that has selection and is admixed from simulated sources related to `{CHB, YRI, IBS}`: (How related the sources are depends on “drift.btwn.surrogates.and.sources” in
CHB selection paramfile.txt, with higher values of this making the simulated sources more different from `{CHB, YRI, IBS}`.)

In [ ]:
R < AdaptMixSimulator.R CHB_selection_paramfile.txt simexample/PEL_REFs_ALLCHR_chr.txt \
simexample/PEL_REFs.ids.txt CHB_selection_ALLCHR --no-save > screenoutput.out

**Question**
 - This simulates selection in CHB before testing for it. Why is it useful to run the test on data where you know the answer?

This will simulate input data to be read into `run_AdaptMix.R` that consists of the real data for the surrogate populations, added atop a simulated `PEL` population. While nearly all SNPs are neutral, one randomly selected SNP – with starting frequency ≥0.05 and ≤0.1 (`range.startfrequency.selected.snp:0.05 0.1`) in the population undergoing selection (`CHB`) – will have strong selection (`sel.coeff: 0.1` per generation, for 150 generations) occurring prior to admixture. This selected SNP will be the last SNP in the output file `CHB_selection_ALLCHR.haps`.

Next run run AdaptMix on this simulated dataset, testing for selection in the simulated PEL population:

In [ ]:
R < run_AdaptMix.R example/PEL_analysis_paramfile.txt CHB_selection_ALLCHR.txt \
CHB_selection_ALLCHR.idfile.txt CHB_selection_ALLCHR.adaptmix.txt --no-save > screenoutput.out2

**Questions**
 - AdaptMix compares the ancestry proportion at each locus with the genome-wide average. Why would selection after admixture produce a departure?
 - Are the top-scoring loci convincing, or could drift produce them?

The output will be in `CHB_selection_ALLCHR.adaptmix.txt`, with scores for the selected SNP in the last row of this file. The header is in the third row, with columns giving the p-value of the selection test (column 3) and other information, such as AIC scores.

Repeat this for another simulation described in `PEL_selection_paramfile.txt`, which instead simulates selection post-admixture, with selection strength s = 0.15 for 50 generations:

In [ ]:
R < AdaptMixSimulator.R PEL_selection_paramfile.txt simexample/PEL_REFs_ALLCHR_chr.txt \
simexample/PEL_REFs.ids.txt PEL_selection_ALLCHR --no-save > screenoutput.out

R < run_AdaptMix.R example/PEL_analysis_paramfile.txt PEL_selection_ALLCHR.txt \
CHB_selection_ALLCHR.idfile.txt PEL_selection_ALLCHR.adaptmix.txt --no-save > screenoutput.out2

**Question**
 - This run simulates selection in PEL rather than CHB. Does AdaptMix recover the locus it was told to select on?

The `AdaptMix` output for this run will be in `PEL_selection_ALLCHR.adaptmix.txt`. 

Use the two `AdaptMix` output files for these two simulations to answer the following questions.
1. For each simulation scenario, is there evidence of selection at the SNP with simulated selection?
2. For the SNP with simulated selection in each scenario, do the results indicate selection post-admixture, or in a particular source population pre-admixture?
3. Looking at the bottom of screenoutput.out, how well do the correlations between the simulated allele frequencies of the sources and their respective surrogate populations match that observed in the real data? How would you adjust drift.btwn.surrogates.and.sources in the input parameter files to make a better match?

In [ ]:
# look at the screen output of the run
tail -n 30 screenoutput.out


**Questions**
 - Comparing all four tools: which agreed on the 30-generation date, and which did not?
 - If you had real data with no known answer, which result would you report, and what would you say about the uncertainty?

### Run the cell below to take the quiz

In [ ]:
from jupyterquiz import display_quiz

display_quiz("https://raw.githubusercontent.com/popgenDK/courses/main/current_exercises/admixture/quiz/dating_admixture.json")
